# 03 — Bi-encoder fine-tuning

Seeded MNRL training is gated on real graded labels and explicit model-download permission. Checkpoints are cached under `artifacts/model/`. In this smoke run, training and comparison are skipped because all supplied labels are blank; synthetic data is never used as training evidence.

In [ ]:
import importlib.util, json, os, random, subprocess, sys
from pathlib import Path
_REQUIRED = {"pandas": "pandas", "pyarrow": "pyarrow"}
_missing = [package for module, package in _REQUIRED.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])
import pandas as pd
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find shared config.json from the notebook working directory")
CONFIG = json.loads((ROOT / "config.json").read_text(encoding="utf-8"))
SEED = int(CONFIG["random_seed"])
random.seed(SEED)
for key in ("data_dir", "artifact_dir", "results_dir"):
    (ROOT / CONFIG[key]).mkdir(parents=True, exist_ok=True)
ARTIFACTS = ROOT / CONFIG["artifact_dir"]
RESULTS = ROOT / CONFIG["results_dir"]
MODEL_DIR = ARTIFACTS / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"Setup complete; smoke_test_mode={CONFIG['smoke_test_mode']}; seed={SEED}")

In [ ]:
import json

policies_df = pd.read_parquet(ARTIFACTS / "policies.parquet")
queries_df = pd.read_csv(ARTIFACTS / "queries.csv")
labels_df = pd.read_parquet(ARTIFACTS / "relevance_labels.parquet")
splits = json.loads((ARTIFACTS / "splits.json").read_text(encoding="utf-8"))
query_by_id = dict(zip(queries_df["query_id"], queries_df["raw_query"]))
policy_text = dict(zip(policies_df["policy_id"], policies_df["policy_name"] + " " + policies_df["category"] + " " + policies_df["coverage_type"] + " " + policies_df["description"]))
train_groups = set(splits["group_split"]["groups"]["train"])
validation_groups = set(splits["group_split"]["groups"]["validation"])
test_groups = set(splits["group_split"]["groups"]["test"])
held_out_policies = set(splits["policy_held_out"]["held_out_policy_ids"])
training_policy_ids = set(splits["policy_held_out"]["training_policy_ids"])

if not labels_df.empty:
    labels_df = labels_df.loc[labels_df["grade"].notna()].copy()
    if "group_id" not in labels_df:
        raise ValueError("Human labels require group_id for leakage-safe training")
    if not set(labels_df["group_id"]).issubset(set(queries_df["group_id"])):
        raise ValueError("Human label group_id does not match an exported query group")
    if not set(labels_df["query_id"]).issubset(set(queries_df["query_id"])):
        raise ValueError("Human labels contain unknown query_id values")
    if not set(labels_df["policy_id"]).issubset(set(policies_df["policy_id"])):
        raise ValueError("Human labels contain unknown policy_id values")

human_rows = int(len(labels_df))
positive_rows = labels_df.loc[labels_df["grade"] >= 2] if human_rows else labels_df
manifest = {
    "mode": "human_label_training" if human_rows else "smoke_test_only",
    "status": None,
    "human_graded_rows": human_rows,
    "positive_training_examples": 0,
    "trained_on_synthetic_labels": False,
    "model_load_included_in_latency": None,
    "experiments": {},
}

if not human_rows:
    manifest["status"] = "SKIPPED_NO_HUMAN_GRADED_LABELS"
    manifest["reason"] = "The provided CSV is a blank template; no relevance grades are available for MNRL training or evaluation."
elif CONFIG["smoke_test_mode"] or not CONFIG["allow_model_downloads"]:
    manifest["status"] = "SKIPPED_SMOKE_MODE"
    manifest["reason"] = "Smoke mode disables pretrained-model downloads and training. Set both flags explicitly after collecting real grades."
elif positive_rows.empty:
    manifest["status"] = "SKIPPED_NO_POSITIVE_TRAINING_PAIRS"
    manifest["reason"] = "No grade >= 2 training pairs are available for MNRL."
else:
    required = ("sentence_transformers", "torch")
    missing = [name for name in required if __import__("importlib").util.find_spec(name) is None]
    if missing:
        manifest["status"] = "SKIPPED_MISSING_TRAINING_DEPENDENCIES"
        manifest["missing_dependencies"] = missing
    else:
        from sentence_transformers import InputExample, SentenceTransformer, losses
        from torch.utils.data import DataLoader

        def train_checkpoint(experiment, allowed_policies):
            checkpoint = MODEL_DIR / experiment
            if (checkpoint / "modules.json").is_file():
                return {"status": "loaded_cached_checkpoint", "path": str(checkpoint.relative_to(ROOT))}
            rows = positive_rows.loc[
                positive_rows["group_id"].isin(train_groups)
                & positive_rows["policy_id"].isin(allowed_policies)
            ]
            examples = [
                InputExample(texts=[query_by_id[row.query_id], policy_text[row.policy_id]])
                for row in rows.itertuples(index=False)
            ]
            if len(examples) < 2:
                return {"status": "skipped_too_few_training_pairs", "positive_pairs": len(examples)}
            model = SentenceTransformer(CONFIG["embedding_model_name"])
            loader = DataLoader(examples, shuffle=True, batch_size=min(8, len(examples)))
            loss = losses.MultipleNegativesRankingLoss(model)
            model.fit(train_objectives=[(loader, loss)], epochs=1, warmup_steps=0, output_path=str(checkpoint), show_progress_bar=False)
            return {"status": "trained", "path": str(checkpoint.relative_to(ROOT)), "positive_pairs": len(examples), "epochs": 1}

        manifest["experiments"]["group_split"] = train_checkpoint("finetuned_group_split", set(policies_df["policy_id"]))
        manifest["experiments"]["policy_held_out"] = train_checkpoint("finetuned_policy_held_out", training_policy_ids)
        manifest["positive_training_examples"] = int(len(positive_rows.loc[positive_rows["group_id"].isin(train_groups)]))
        manifest["status"] = "COMPLETED" if all(v["status"] in ("trained", "loaded_cached_checkpoint") for v in manifest["experiments"].values()) else "PARTIAL"

metadata = {
    "seed": SEED,
    "train_group_count": len(train_groups),
    "validation_group_count": len(validation_groups),
    "test_group_count": len(test_groups),
    "policy_held_out_count": len(held_out_policies),
    "policy_training_count": len(training_policy_ids),
    "base_encoder": CONFIG["embedding_model_name"],
    "fine_tuning_loss": "MultipleNegativesRankingLoss",
    "cache_directory": MODEL_DIR.relative_to(ROOT).as_posix(),
    "experiment_results": manifest,
}
(MODEL_DIR / "manifest.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
(RESULTS / "finetune.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))

## Assertions

In [ ]:
assert not (train_groups & validation_groups)
assert not (train_groups & test_groups)
assert not (validation_groups & test_groups)
assert training_policy_ids.isdisjoint(held_out_policies)
assert manifest["trained_on_synthetic_labels"] is False
if human_rows == 0:
    assert manifest["status"] == "SKIPPED_NO_HUMAN_GRADED_LABELS"
    assert manifest["positive_training_examples"] == 0
    assert not manifest["experiments"]
assert json.loads((RESULTS / "finetune.json").read_text(encoding="utf-8"))["status"] == manifest["status"]
print("Notebook 03 checks passed: group isolation, policy holdout, cache manifest, and no synthetic-label training.")